# perturb-data-lab Demo Walkthrough

This notebook walks through the full demo workflow from HuggingFace download
through canonicalized corpus and pertTF loader.

## Prerequisites

- Installation completed: `pip install -e ".[demo]"`
- Working Python environment with `perturb_data_lab` importable

In [ ]:
# Cell 1 -- Download demo data
import subprocess
import sys
from pathlib import Path

result = subprocess.run(
    [sys.executable, "scripts/download_demo_data.py", "--output-dir", "./demo_data"],
    capture_output=False,
)

demo_root = Path("./demo_data")
for rel in ["h5ad/demo_marson_d2_rest.h5ad", "h5ad/demo_xorion_hct116_dual_guide.h5ad"]:
    f = demo_root / rel
    size_mb = f.stat().st_size / 1e6 if f.exists() else 0
    print(f"  {rel}: {size_mb:.1f} MB {'(ok)' if f.exists() else 'MISSING'}")

## Cell 2 -- Inspect raw h5ad metadata

Inspection reads metadata and samples matrix candidates without loading full
count matrices.

In [ ]:
from perturb_data_lab.inspectors import inspect_target
from perturb_data_lab.inspectors.models import InspectionTarget

review_dir = Path("./artifacts/review")
review_dir.mkdir(parents=True, exist_ok=True)

datasets = [
    ("marson_d2_rest", "./demo_data/h5ad/demo_marson_d2_rest.h5ad"),
    ("xorion_hct116_dual_guide", "./demo_data/h5ad/demo_xorion_hct116_dual_guide.h5ad"),
]

for ds_id, source_path in datasets:
    result = inspect_target(
        InspectionTarget(dataset_id=ds_id, source_path=source_path, source_release=ds_id),
        review_dir,
    )
    print(f"\n{ds_id}:")
    print(f"  cells: {result.n_obs}, features: {result.n_vars}")
    print(f"  count source: {result.count_source}")
    print(f"  readiness: {result.materialization_readiness}")

## Cell 3 -- Quick peek at raw metadata

In [ ]:
import anndata

for ds_id, source_path in datasets:
    adata = anndata.read_h5ad(source_path, backed="r")
    print(f"\n--- {ds_id} ---")
    print(f"Shape: {adata.shape}")
    cols = [c for c in ["guide_id", "guide_type", "perturbed_gene_name",
                        "guide_target", "gene_target", "sample"]
            if c in adata.obs.columns]
    if cols:
        print(adata.obs[cols].head(5).to_string())

## Cell 4 -- Materialize (create mode)

Create aggregate Lance corpus from the first dataset.

In [ ]:
from perturb_data_lab.materializers import DatasetMaterializer
from perturb_data_lab.materializers.models import OutputRoots
from perturb_data_lab.materializers.paths import resolve_corpus_paths
import yaml

corpus_root = Path("./artifacts/demo_corpus")
corpus_root.mkdir(parents=True, exist_ok=True)


def load_summary(dataset_id):
    path = Path("./artifacts/review") / dataset_id / "dataset-summary.yaml"
    return yaml.safe_load(path.read_text())


ds_id = "marson_d2_rest"
paths = resolve_corpus_paths("aggregate", corpus_root, ds_id)
summary = load_summary(ds_id)

m = DatasetMaterializer(
    source_path="./demo_data/h5ad/demo_marson_d2_rest.h5ad",
    inspection_summary_path=str(
        Path("./artifacts/review") / ds_id / "dataset-summary.yaml"
    ),
    output_roots=OutputRoots(
        metadata_root=str(paths.meta_root),
        matrix_root=str(paths.matrix_root),
    ),
    dataset_id=ds_id,
    backend="lance",
    topology="aggregate",
    corpus_index_path=str(corpus_root / "corpus-index.yaml"),
    corpus_id="demo_corpus",
    register=True,
    mode="create",
    dataset_index=0,
    global_row_start=0,
)
manifest = m.materialize()
print(f"Created {ds_id}: {manifest.cell_count} cells, "
      f"{manifest.feature_count} features")

## Cell 5 -- Append second dataset

In [ ]:
ds_id = "xorion_hct116_dual_guide"
paths = resolve_corpus_paths("aggregate", corpus_root, ds_id)
summary = load_summary(ds_id)

index_doc = yaml.safe_load((corpus_root / "corpus-index.yaml").read_text())
first_entry = index_doc["datasets"][0]
next_global_start = first_entry["global_end"]
next_dataset_index = first_entry["dataset_index"] + 1

m2 = DatasetMaterializer(
    source_path="./demo_data/h5ad/demo_xorion_hct116_dual_guide.h5ad",
    inspection_summary_path=str(
        Path("./artifacts/review") / ds_id / "dataset-summary.yaml"
    ),
    output_roots=OutputRoots(
        metadata_root=str(paths.meta_root),
        matrix_root=str(paths.matrix_root),
    ),
    dataset_id=ds_id,
    backend="lance",
    topology="aggregate",
    corpus_index_path=str(corpus_root / "corpus-index.yaml"),
    corpus_id="demo_corpus",
    register=True,
    mode="append",
    dataset_index=next_dataset_index,
    global_row_start=next_global_start,
)
manifest2 = m2.materialize()
print(f"Appended {ds_id}: {manifest2.cell_count} cells, "
      f"{manifest2.feature_count} features")

## Cell 6 -- Install reviewed schemas

In [ ]:
subprocess.run(
    [sys.executable, "scripts/install_demo_schemas.py", "--corpus", str(corpus_root)],
    check=True,
)

for ds_id in ["marson_d2_rest", "xorion_hct116_dual_guide"]:
    schema_path = corpus_root / "meta" / ds_id / "final-schema.yaml"
    print(f"  {ds_id}: {'ok' if schema_path.exists() else 'MISSING'}")

## Cell 7 -- Canonicalize

In [ ]:
subprocess.run(
    [sys.executable, "-m", "perturb_data_lab.cli", "canonicalize",
     "--corpus", str(corpus_root), "--dry-run"],
    check=True,
)
print("Dry-run passed.")

subprocess.run(
    [sys.executable, "-m", "perturb_data_lab.cli", "canonicalize",
     "--corpus", str(corpus_root)],
    check=True,
)
print("Canonicalization complete.")

## Cell 8 -- Validate and load the corpus

In [ ]:
subprocess.run(
    [sys.executable, "-m", "perturb_data_lab.cli", "corpus-validate",
     str(corpus_root / "corpus-index.yaml")],
    check=True,
)

from perturb_data_lab.loaders import load_corpus

corpus = load_corpus(str(corpus_root))
print(f"Datasets: {corpus.dataset_ids}")
print(f"Total cells: {len(corpus.metadata_index)}")
print(f"Global vocab size: {corpus.feature_registry.global_vocab_size}")

## Cell 9 -- Inspect canonical metadata

In [ ]:
import polars as pl

marson_rows = corpus.take_metadata(
    list(range(0, 10)),
    columns=["dataset_id", "perturb_label", "condition",
             "perturb_type", "cell_context", "batch_id"],
)
xorion_rows = corpus.take_metadata(
    list(range(2720, 2730)),
    columns=["dataset_id", "perturb_label", "condition",
             "perturb_type", "cell_context", "batch_id"],
)

print("--- Marson ---")
print(marson_rows)
print("\n--- Xorion ---")
print(xorion_rows)

meta = corpus.metadata_index
all_labels = meta.get_column("perturb_label")
ctrl_count = (pl.Series(all_labels) == "ctrl").sum()
total = len(all_labels)
print(f"\nControl rows: {ctrl_count} / {total} "
      f"({100 * ctrl_count / total:.0f}%)")

## Cell 10 -- PertTF loader preview

In [ ]:
from perturb_data_lab.loaders import (
    PertTFAdapterConfig,
    PertTFPairedBatchLoader,
)

config = PertTFAdapterConfig(
    label_fields={
        "perturb_label": "perturbation",
        "cell_context": "celltype",
        "batch_id": "batch",
        "dataset_index": "dataset",
    },
    perturbation_label="perturbation",
    control_labels=("ctrl",),
    pairing_group_labels=("dataset", "celltype"),
)

loader = PertTFPairedBatchLoader(
    corpus,
    batch_size=4,
    seq_len=64,
    config=config,
    sampling_mode="hvg",
    hvg_top_k=2000,
    num_workers=0,
)

batch = next(iter(loader))
print("Batch keys:", sorted(batch.keys()))
print(f"  gene_ids shape: {batch['gene_ids'].shape}")
print(f"  values shape: {batch['values'].shape}")
print(f"  target_values shape: {batch['target_values'].shape}")
print(f"  target_values_next shape: {batch['target_values_next'].shape}")

src_labels = corpus.take_metadata(
    batch["index"].tolist(),
    columns=["perturb_label", "dataset_id"],
)
tgt_labels = corpus.take_metadata(
    batch["next_index"].tolist(),
    columns=["perturb_label", "dataset_id"],
)
print("\nSource -> Target pairs:")
for i in range(len(batch["index"])):
    src = src_labels.row(i)
    tgt = tgt_labels.row(i)
    print(f"  {src[1]} {src[0]} -> {tgt[1]} {tgt[0]}")

## Cell 11 -- AnnData handoff (Dask-backed, inner join)

In [ ]:
adata = corpus.to_anndata_lazy(
    dataset_id=list(corpus.dataset_ids),
    obs_columns=["perturb_label", "condition", "cell_context", "batch_id"],
    chunk_rows=1024,
    var_join="inner",
)

print(f"adata shape: {adata.shape}")
print(f"adata.X type: {type(adata.X)}")
print(f"Intersection features: {adata.n_vars}")
print(f"obs columns: {list(adata.obs.columns)}")

## Cell 12 -- Quick Scanpy smoke

In [ ]:
import scanpy as sc

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
print(f"HVG selected: {adata.var['highly_variable'].sum()}")

subset = adata[:100, :100].to_memory()
print(f"Subset shape: {subset.shape}")